In [ ]:
import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.cluster.hierarchy import cophenet, dendrogram, linkage
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

import brain_probability_flux.plotting as bpp

In [ ]:
# Results are read from data/. To plot a fresh run, set DATA_DIR to its run
# directory, for example Path("../data/2026-10-20").
DATA_DIR = bpp.DATA_DIR

In [ ]:
# This figure is dense enough that the project default (7pt) causes
# text/content overlap and excess empty space; use 5pt here instead.
plt.rcParams["font.size"] = 5.0

In [ ]:
max_fig_width = bpp.FIG_WIDTH
max_fig_height = bpp.FIG_HEIGHT

task_labels = bpp.TASK_NAMES_FULL
NUM_TASKS = 8
NUM_CLUSTERS = 7
save_dir = "../figures/fig-05"

In [ ]:
beta_J = bpp.load_results("inferred_networks", "arrhenius", DATA_DIR)[
    "beta_J"
]  # (8, 7, 7)
upper = np.triu_indices(NUM_CLUSTERS, k=1)
off_diagonal = ~np.eye(NUM_CLUSTERS, dtype=bool)
# Upper triangle suffices for the symmetric and antisymmetric parts; the
# full J needs all off-diagonal elements (row-major).
vecs_full = np.array([J[off_diagonal] for J in beta_J])  # (8, 42)
vecs_sym = np.array([bpp.get_symmetric(J)[upper] for J in beta_J])  # (8, 21)
vecs_anti = np.array(
    [bpp.get_antisymmetric(J)[upper] for J in beta_J]
)  # (8, 21)


def cosine(u: np.ndarray, v: np.ndarray) -> float:
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

## Panels a-c — task similarity of $\boldsymbol{J}$, $\boldsymbol{J}^{(s)}$ and $\boldsymbol{J}^{(a)}$

Pairwise cosine similarity across the 8 conditions of the vectorized
couplings, clustered by average linkage of the cosine distance.

In [ ]:
def plot_task_similarity(vecs, title, label):
    cos = cosine_similarity(vecs)
    Z = linkage(
        pdist(vecs, metric="cosine"), method="average", optimal_ordering=True
    )
    r_coph, _ = cophenet(Z, pdist(vecs, metric="cosine"))

    plt.rcParams["figure.constrained_layout.use"] = False
    g = sns.clustermap(
        cos,
        row_linkage=Z,
        col_linkage=Z,
        xticklabels=task_labels,
        yticklabels=task_labels,
        cmap=cc.cm.CET_D1A,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt=".2f",
        cbar_kws={"label": "Cos. sim.", "shrink": 0.5},
        figsize=(max_fig_width / 3, 63 / 25.4),
        rasterized=True,
    )
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), rotation=90)
    plt.rcParams["figure.constrained_layout.use"] = True

    # subplots_adjust to pack tick labels within figsize + fix colorbar alignment
    _fw, _fh = g.figure.get_size_inches()
    _wp = plt.rcParams.get("figure.constrained_layout.w_pad", 0.025002)
    _hp = plt.rcParams.get("figure.constrained_layout.h_pad", 0.025002)
    g.figure.canvas.draw()
    renderer = g.figure.canvas.get_renderer()
    _ax = g.ax_heatmap
    _tick_ys = [
        tl.get_window_extent(renderer).y0 for tl in _ax.get_xticklabels()
    ]
    _xtick_in = (
        _ax.get_position().y0 * _fh * g.figure.dpi - min(_tick_ys)
    ) / g.figure.dpi
    _tick_xs = [
        tl.get_window_extent(renderer).x1 for tl in _ax.get_yticklabels()
    ]
    _ytick_in = (
        max(_tick_xs) - _ax.get_position().x1 * _fw * g.figure.dpi
    ) / g.figure.dpi
    _new_left = _wp / _fw
    _new_right = 1.0 - (_wp + _ytick_in) / _fw
    _new_bottom = (_hp + _xtick_in) / _fh
    _title_h_in = 8 / 72
    _new_top = 1.0 - (_hp + _title_h_in) / _fh
    _cb0 = g.ax_cbar.get_position()
    _cb_top = _cb0.y0 + _cb0.height
    g.figure.subplots_adjust(
        left=_new_left, right=_new_right, bottom=_new_bottom, top=_new_top
    )
    _col_d_bot = g.ax_col_dendrogram.get_position().y0
    g.ax_cbar.set_position(
        [_new_left, _col_d_bot, _cb0.width, _cb_top - _col_d_bot]
    )
    g.ax_cbar.tick_params(labelsize=5)
    g.ax_cbar.yaxis.label.set_size(5)
    g.figure.text(
        0.5,
        1.0 - _title_h_in / (2 * _fh),
        f"{title}, Cophenetic $r$ = {r_coph:.3f}",
        ha="center",
        va="center",
    )
    bpp.save_panel(g.figure, save_dir, label, bbox_inches="tight")
    return Z


Z_full = plot_task_similarity(vecs_full, "Interaction $\\boldsymbol{J}$", "a")
Z_sym = plot_task_similarity(
    vecs_sym, "Symmetric $\\boldsymbol{J}^\\mathrm{(s)}$", "b"
)
Z_anti = plot_task_similarity(
    vecs_anti, "Antisymmetric $\\boldsymbol{J}^\\mathrm{(a)}$", "c"
)

## Panel d — cosine similarity per task

For each task $k$: cosine similarity of $\boldsymbol{J}^{(s)}_k$ (and
$\boldsymbol{J}^{(a)}_k$) against (1) the leave-one-out task average and
(2) the rest condition. Error bars: standard deviation over the bootstrapped
couplings (scripts/bootstrap_inference.py).

In [ ]:
def similarity_to_rest_and_others(vecs):
    """(cos vs rest, cos vs leave-one-out average) per task."""
    vs_rest = np.array([cosine(vecs[k], vecs[0]) for k in range(NUM_TASKS)])
    vs_avg = np.array(
        [
            cosine(vecs[k], np.delete(vecs, k, axis=0).mean(axis=0))
            for k in range(NUM_TASKS)
        ]
    )
    return vs_rest, vs_avg


beta_J_boot = bpp.load_results("inference_bootstrap", "arrhenius", DATA_DIR)[
    "beta_J"
]  # (8, n_boot, K, K)
n_boot = beta_J_boot.shape[1]
J_t = beta_J_boot.transpose(0, 1, 3, 2)
vecs_sym_boot = (0.5 * (beta_J_boot + J_t))[
    :, :, upper[0], upper[1]
]  # (8, n_boot, 21)
vecs_anti_boot = (0.5 * (beta_J_boot - J_t))[:, :, upper[0], upper[1]]

similarity = {}
for name, vecs, vecs_boot in [
    ("sym", vecs_sym, vecs_sym_boot),
    ("anti", vecs_anti, vecs_anti_boot),
]:
    boot = np.array(
        [similarity_to_rest_and_others(vecs_boot[:, b]) for b in range(n_boot)]
    )
    similarity[name] = (
        *similarity_to_rest_and_others(vecs),
        *boot.std(axis=0),
    )

In [ ]:
fig, axs = plt.subplots(
    1, 2, figsize=(max_fig_width / 3, 40 / 25.4), sharey=True
)
x = np.arange(NUM_TASKS)
width = 0.35
ebar_kw = {
    "fmt": "none",
    "color": "k",
    "capsize": 2,
    "elinewidth": plt.rcParams["xtick.major.width"],
    "capthick": plt.rcParams["xtick.major.width"],
}
for ax, name, title in zip(
    axs,
    ["sym", "anti"],
    [
        r"Symmetric $\boldsymbol{J}^\mathrm{(s)}$",
        r"Antisymmetric $\boldsymbol{J}^\mathrm{(a)}$",
    ],
):
    vs_rest, vs_avg, vs_rest_std, vs_avg_std = similarity[name]
    ax.bar(x - width / 2, vs_rest, width, label="vs rest", color="0.5")
    ax.bar(
        x + width / 2,
        vs_avg,
        width,
        label="vs task-average (LOO)",
        color="0.85",
    )
    ax.errorbar(x - width / 2, vs_rest, yerr=vs_rest_std, **ebar_kw)
    ax.errorbar(x + width / 2, vs_avg, yerr=vs_avg_std, **ebar_kw)
    ax.set_xticks(x)
    ax.set_xticklabels(task_labels, rotation=90)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_title(title)
    ax.legend(frameon=False)
axs[0].set_ylabel("Cosine similarity")
axs[0].set_ylim(-1.1, 1.1)
bpp.save_panel(fig, save_dir, "d", bbox_inches="tight")

## Panel e — dendrogram comparison with the probability-flux task clustering

Baker's gamma: Spearman rank correlation between the cophenetic distances
of each coupling dendrogram and of the flux dendrogram (Fig. 3a).

In [ ]:
pfa, _ = bpp.aggregated_pfa(data_dir=DATA_DIR)
flux_flat, _, _ = bpp.observed_edge_flux(pfa)  # (8, n_transitions)
dist_flux = pdist(flux_flat, metric="cosine")
_, coph_flux = cophenet(
    linkage(dist_flux, method="average", optimal_ordering=True), dist_flux
)

fig, axs = plt.subplots(
    1, 3, figsize=(max_fig_width / 2, 40 / 25.4), sharey=True
)
for ax, Z, vecs, title in zip(
    axs,
    [Z_full, Z_sym, Z_anti],
    [vecs_full, vecs_sym, vecs_anti],
    [
        r"Interaction $\boldsymbol{{J}}$, Baker's $\gamma$={:.3f}",
        r"Symmetric $\boldsymbol{{J}}^\mathrm{{(s)}}$, Baker's $\gamma$={:.3f}",
        r"Antisymmetric $\boldsymbol{{J}}^\mathrm{{(a)}}$, Baker's $\gamma$={:.3f}",
    ],
):
    _, coph = cophenet(Z, pdist(vecs, metric="cosine"))
    baker_gamma, _ = spearmanr(coph, coph_flux)
    dendrogram(
        Z,
        labels=task_labels,
        ax=ax,
        color_threshold=0,
        above_threshold_color="k",
        leaf_rotation=90,
        leaf_font_size=5,
    )
    ax.set_title(title.format(baker_gamma))
axs[0].set_ylabel("Distance")
bpp.save_panel(fig, save_dir, "e", bbox_inches="tight")

## Panel f — asymmetry ratio $\|\boldsymbol{J}^{(a)}\|_F / \|\boldsymbol{J}^{(s)}\|_F$ per task

In [ ]:
asym_ratio = bpp.asymmetry_ratio(beta_J)
asym_ratio_std = bpp.asymmetry_ratio(beta_J_boot).std(axis=1)

fig, ax = plt.subplots(figsize=(max_fig_width / 6, 40 / 25.4))
x = np.arange(NUM_TASKS)
ax.bar(x, asym_ratio, color="0.75")
ax.errorbar(x, asym_ratio, yerr=asym_ratio_std, **ebar_kw)
ax.set_xticks(x)
ax.set_xticklabels(task_labels, rotation=90)
ax.set_ylabel(
    r"$\|\boldsymbol{J}^{(\mathrm{a})}\|_{\mathrm{F}}\,/\,\|\boldsymbol{J}^{(\mathrm{s})}\|_{\mathrm{F}}$"
)
ax.axhline(
    asym_ratio.mean(),
    color="k",
    ls="--",
    lw=plt.rcParams["xtick.major.width"],
    label=f"Mean = {asym_ratio.mean():.3f}",
)
ax.legend(frameon=False)
ax.set_ylim(top=0.28)
bpp.save_panel(fig, save_dir, "f", bbox_inches="tight")